# 06 - Model Evaluation

Comprehensive evaluation of the best-performing model.

## Objectives:
- Load the saved best model
- Evaluate on test set (MAE, RMSE, MAPE)
- Performance by store, category, department
- Error analysis and diagnostics
- Visual comparison of all models

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.forecasting import DemandForecaster
from src.utils import load_dataframe, load_model

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Load Best Model and Data

In [ ]:
forecaster = DemandForecaster()
forecaster.load_model('../models/best_model.joblib')

df = load_dataframe('../data/processed/engineered_features.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Model type: {forecaster.model_type}")
print(f"Data shape: {df.shape}")
print(f"Features used: {len(forecaster.feature_names)}")

## 2. Model Comparison Table

Review the saved comparison results from training.

In [ ]:
try:
    comparison = pd.read_csv('../reports/exports/model_comparison.csv')
    print("Model Comparison (all models):")
    print("=" * 70)
    display(comparison.round(4))
    
    best_row = comparison.dropna(subset=['MAE']).loc[comparison['MAE'].idxmin()]
    print(f"\nBest model: {best_row['model']} (MAE: {best_row['MAE']:.4f}, MAPE: {best_row['MAPE']:.2f}%)")
except FileNotFoundError:
    print("Model comparison not found. Run 05_Model_Training first.")

## 3. Evaluate on Test Set (Last 28 Days)

In [ ]:
# Split by date
max_date = df['date'].max()
test_start = max_date - pd.Timedelta(days=28)
test_df = df[df['date'] > test_start].copy()
train_df = df[df['date'] <= test_start].copy()

print(f"Train period: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Test period:  {test_df['date'].min()} to {test_df['date'].max()}")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")

# Evaluate
X_test, y_test = forecaster.prepare_features(test_df, target_col='sales')
metrics = forecaster.evaluate(X_test, y_test)

print("\n" + "=" * 50)
print("BEST MODEL — TEST SET PERFORMANCE")
print("=" * 50)
for metric, value in metrics.items():
    print(f"  {metric:15s}: {value:10.4f}")
print("=" * 50)

## 4. Performance by Store

In [ ]:
# Generate predictions
predictions = forecaster.predict(X_test)
test_df_eval = test_df.copy()
test_df_eval['predicted_sales'] = predictions
test_df_eval['abs_error'] = np.abs(test_df_eval['sales'] - predictions)

store_perf = test_df_eval.groupby('store_id').agg({
    'sales': 'sum',
    'predicted_sales': 'sum',
    'abs_error': 'mean'
}).reset_index()
store_perf['mape'] = np.abs(
    (store_perf['sales'] - store_perf['predicted_sales']) / store_perf['sales'].replace(0, np.nan)
) * 100

print("Performance by Store:")
print(store_perf.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(store_perf['store_id'], store_perf['abs_error'], color='#2C5282')
axes[0].set_title('Mean Absolute Error by Store')
axes[0].set_ylabel('MAE')
axes[1].bar(store_perf['store_id'], store_perf['mape'], color='#E53E3E')
axes[1].set_title('MAPE by Store')
axes[1].set_ylabel('MAPE %')
plt.tight_layout()
plt.show()

## 5. Performance by Category

In [ ]:
cat_perf = test_df_eval.groupby('cat_id').agg({
    'sales': 'sum',
    'predicted_sales': 'sum',
    'abs_error': 'mean'
}).reset_index()
cat_perf['mape'] = np.abs(
    (cat_perf['sales'] - cat_perf['predicted_sales']) / cat_perf['sales'].replace(0, np.nan)
) * 100

print("Performance by Category:")
print(cat_perf.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(cat_perf['cat_id'], cat_perf['abs_error'], color='#38A169')
axes[0].set_title('MAE by Category')
axes[1].bar(cat_perf['cat_id'], cat_perf['mape'], color='#D69E2E')
axes[1].set_title('MAPE by Category')
plt.tight_layout()
plt.show()

## 6. Error Analysis

In [ ]:
errors = test_df_eval['sales'] - predictions
abs_errors = np.abs(errors)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Error distribution
axes[0, 0].hist(errors, bins=50, edgecolor='black', color='#2C5282')
axes[0, 0].set_title('Error Distribution')
axes[0, 0].set_xlabel('Error')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(x=0, color='red', linestyle='--')

# Actual vs Predicted
axes[0, 1].scatter(test_df_eval['sales'], predictions, alpha=0.3, s=1, color='#2C5282')
max_val = max(test_df_eval['sales'].max(), predictions.max())
axes[0, 1].plot([0, max_val], [0, max_val], 'r--', label='Perfect')
axes[0, 1].set_title('Actual vs Predicted')
axes[0, 1].set_xlabel('Actual Sales')
axes[0, 1].set_ylabel('Predicted Sales')
axes[0, 1].legend()

# Residuals
axes[1, 0].scatter(predictions, errors, alpha=0.3, s=1, color='#E53E3E')
axes[1, 0].axhline(y=0, color='black', linestyle='--')
axes[1, 0].set_title('Residual Plot')
axes[1, 0].set_xlabel('Predicted Sales')
axes[1, 0].set_ylabel('Residuals')

# Abs Error by Actual
axes[1, 1].scatter(test_df_eval['sales'], abs_errors, alpha=0.3, s=1, color='#38A169')
axes[1, 1].set_title('Absolute Error by Actual Sales')
axes[1, 1].set_xlabel('Actual Sales')
axes[1, 1].set_ylabel('Absolute Error')

plt.tight_layout()
plt.savefig('../reports/figures/evaluation_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Diagnostics saved to reports/figures/evaluation_diagnostics.png")

## 7. Worst Predictions

In [ ]:
test_df_eval['error'] = errors
test_df_eval['abs_error'] = abs_errors

worst = test_df_eval.nlargest(20, 'abs_error')[
    ['item_id', 'store_id', 'date', 'sales', 'predicted_sales', 'error', 'abs_error']
]

print("Top 20 Worst Predictions:")
print(worst.to_string(index=False))
print(f"\nStats on worst 20:")
print(f"  Avg actual: {worst['sales'].mean():.2f}")
print(f"  Avg predicted: {worst['predicted_sales'].mean():.2f}")
print(f"  Most common store: {worst['store_id'].mode()[0]}")

## 8. Sample Forecast Visualization

In [ ]:
sample_item = test_df_eval['item_id'].iloc[0]
sample_store = test_df_eval['store_id'].iloc[0]

sample_data = test_df_eval[
    (test_df_eval['item_id'] == sample_item) & 
    (test_df_eval['store_id'] == sample_store)
].sort_values('date')

plt.figure(figsize=(12, 5))
plt.plot(sample_data['date'], sample_data['sales'], 'b-', label='Actual', linewidth=2)
plt.plot(sample_data['date'], sample_data['predicted_sales'], 'r--', label='Predicted', linewidth=2)
plt.title(f'Sales Forecast for {sample_item} at {sample_store}')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Summary

Model evaluation completed:
- Best model identified from comparison
- Test set performance measured
- Performance by store analyzed
- Performance by category analyzed
- Error distribution examined
- Worst predictions identified
- Diagnostics saved

Next: SHAP Analysis